# Estimation: Kriging, Inverse Distance Weighting (IDW) and K-Nearest Neighbors (KNN)

This notebook demonstrates how to perform estimation using Seequent Evo's Geostatistical Compute tasks.

Before running this notebook, create a Seequent Evo workspace and populate it with the example data using [1-populate-workspace.ipynb](./1-populate-workspace.ipynb).

---

First, import a few python packages.

In [ ]:
import pandas as pd
import plotting
from dotenv import dotenv_values

import evo.objects.typed as objs
from evo.compute import tasks
from evo.compute.tasks.geostatistics import idw, knn, kriging
from evo.notebooks import ServiceManagerWidget

%load_ext evo.widgets

## Authenticate
Log into Seequent Evo and select the appropriate workspace. It is assumed that prior to running this example notebook a Seequent Evo workspace has been created and populated with the necessary example data. If not, refer to the prior notebook [1-populate-workspace.ipynb`](./1-populate-workspace.ipynb)

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(**dotenv_values()).login()

## Load the Objects Needed

Note that when accessing an object from the Evo workspace, a reference to that object is returned.

This reference contains information about the unique ID and location of that object as well as the coordinates and attributes. What this object reference does **not** contain is the actual data. 

The object reference and the tabular data of the object are separated such that workflows can be orchestrated on Evo without ever having to download the data locally. However, if access to the underlying data is desired, use the `.to_dataframe` method as needed to access the numerical data associated with the object.

In this case, a pointset and the blockmodel are needed.

In [ ]:
comps = await objs.object_from_path(evo_connection, "WP comps.json")
variogram = await objs.object_from_path(evo_connection, "Cu_pct Variogram Model.json")
block_model = await objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

Each object, such as the variogram, can be inspected via its html representation, which includes linkes to the `Portal` and the `Viewer`.

In [ ]:
variogram

To inspect the variogram in each principal direction, leverage a python based potting package. Extract the information about each principal direction from the variogram model and plot. In this case [plotly](https://docs.plotly.com/) is used here, the code can be seen in [helpers.py](helpers.py)

In [ ]:
fig = plotting.variogram_model(variogram)
fig.show()

## Generate the Estimation Parameters

First, create the search ellipsoid and neighborhood that can be used for multiple estimations. The ellipsoid (called `ell` in this example) is generated from the variogram ranges and orientations, then scaled to be 1.5x the size of the variogram ellipsoid.

### Kriging

In [ ]:
ell = variogram.get_ellipsoid().scaled(1.5)
srch = tasks.SearchNeighborhood(ellipsoid=ell, max_samples=30, min_samples=1)

In [ ]:
early_diorite_source_filter = tasks.Filter(
    where=tasks.FilterCondition(
        attribute=comps.attributes["Geological_model"],
        operator="in",
        values=["Early Diorite"],
    ),
)

early_diorite_target_filter = tasks.Filter(
    where=tasks.FilterCondition(
        attribute=block_model.attributes["Geological model"],
        operator="in",
        values=["Early Diorite"],
    ),
)

In [ ]:
krig_params = kriging.KrigingParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_OK"],
    target_filter=early_diorite_target_filter,
    variogram=variogram,
    neighborhood=srch,
    block_discretisation=kriging.BlockDiscretisation(nx=5, ny=5, nz=2),
    diagnostics=kriging.KrigingDiagnostics(kriging_variance=block_model.attributes["KV"]),
)

#### Optional: request kriging diagnostics

Kriging can write per-location **diagnostics** onto the target object alongside the estimate. Only the diagnostics you ask for are computed, and each one gets its own attribute — so every name must differ from the estimate's attribute name and from the other diagnostics.

Each field of `KrigingDiagnostics` accepts:

- `True` — create an attribute using the recommended name (these match Leapfrog's conventions, so results are familiar once imported).
- a string — create an attribute with that name instead.
- an attribute from the target object — update it if it already exists, otherwise create it.

```python
from evo.compute.tasks import KrigingDiagnostics

diagnostics = KrigingDiagnostics(
    kriging_variance=True,  # creates "KV"
    kriging_efficiency="CU_efficiency",  # creates "CU_efficiency"
    num_samples=block_model.attributes["NS"],  # updates "NS" if it already exists
)
```

| Diagnostic | Recommended name | Description |
| --- | --- | --- |
| `valid` | `valid` | Whether the location's neighbourhood satisfied the search constraints and produced an estimate. |
| `kriging_variance` | `KV` | Kriging variance (estimation uncertainty). |
| `slope_of_regression` | `SoR` | Slope of regression, a diagnostic for conditional bias. |
| `kriging_efficiency` | `KE` | Proportion of point or intra-block variance explained by the kriging weights. |
| `kriging_mean` | `KM` | GLS mean for ordinary kriging, or the supplied constant mean for simple kriging. |
| `num_samples` | `NS` | Number of data samples used in the estimate. |
| `num_drillholes` | `NDh` | Number of distinct drillholes contributing to the estimate. Requires a downhole-intervals source. |
| `num_duplicates` | `ND` | Number of duplicate sample locations used in the estimate. |
| `num_equidistant` | `NeD` | Hint of how many other samples could have replaced the last returned sample because they were the same distance away. |
| `sum_weights` | `Sum` | Sum of the kriging weights applied to data samples. |
| `sum_positive_weights` | `SumP` | Sum of the positive kriging weights. |
| `sum_negative_weights` | `SumN` | Sum of the negative kriging weights. |
| `min_distance` | `MinD` | Minimum isotropic Euclidean distance from the location to any neighbour. |
| `mean_distance` | `AvgD` | Mean isotropic Euclidean distance from the location to all neighbours. |
| `aniso_min_distance` | `MinAD` | Minimum anisotropic (ellipsoid-space) distance from the location to any neighbour. |
| `aniso_mean_distance` | `AvgAD` | Mean anisotropic (ellipsoid-space) distance from the location to all neighbours. |


### Inverse Distance Weighting (IDW)

In [ ]:
idw_params = idw.IDWParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_IDW"],
    target_filter=early_diorite_target_filter,
    neighborhood=srch,
    power=2,
)

### K-Nearest Neighbors (KNN)

In [ ]:
knn_params = knn.KNNParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_KNN"],
    target_filter=early_diorite_target_filter,
    neighborhood=srch,
    k=1,
)

## Execute the Estimation Tasks

Now that parameters for all tasks have been defined, they can be executed concurrently. Again note the `await` used here to execute tasks asyncronously meaning there is no seequence or waiting for each task to finish, they all run simultaneosly and will communicate their status back to the user. As the tasks conclude their computational work, results will be saved to the block model and available on blocksync or can be imported into leapfrog immediately with by reloading the blocksync model.

In [ ]:
results = await tasks.run(evo_connection, [krig_params, idw_params, knn_params], preview=True)
results

Refresh the `block_model` to retrieve the results from blocksync

In [ ]:
updated_block_model = await block_model.refresh()
updated_block_model

To download the results as a dataframe use `.to_dataframe`. Specify column names to avoid downloading more data than necessary.

In [ ]:
block_model_df = await updated_block_model.to_dataframe(columns=["Cu_KNN", "Cu_IDW", "Cu_OK", "Geological model"])
block_model_df.head()

For use in comparison and validation, retrieve the composites as well.

In [ ]:
comps = await comps.refresh()
comps_df = await comps.to_dataframe("Cu_pct", "Geological_model", "IDW declus")
comps_df.head()

For ease of plotting, combine the composites and results into a single dataframe

In [ ]:
composites_df = comps_df.query("Geological_model == 'Early Diorite'")[["Cu_pct", "IDW declus"]].rename(
    columns={"Cu_pct": "Cu (%)", "IDW declus": "weight"}
)
composites_df = composites_df.dropna().query("weight > 0")
composites_df["Method"] = "Declustered Composites"

estimates_df = (
    block_model_df.query("`Geological model` == 'Early Diorite'")[["Cu_KNN", "Cu_IDW", "Cu_OK"]]
    .melt(var_name="Method", value_name="Cu (%)")
    .dropna()
)
estimates_df["weight"] = 1.0

histogram_df = pd.concat([composites_df, estimates_df], ignore_index=True)
histogram_df.head()

In [ ]:
fig = plotting.histogram_comparison(
    histogram_df,
    x="Cu (%)",
    color="Method",
    weights="weight",
    category_order=["Declustered Composites", "Cu_KNN", "Cu_IDW", "Cu_OK"],
    colors=["#071E22", "#1D7874", "#F4C095", "#EE2E31"],
    title="Early Diorite: Declustered Composites vs Estimation Results",
)
fig.show()

Inspect the summary statistics

In [ ]:
from helpers import compare_weighted_summary

composite_summary = compare_weighted_summary(
    values=composites_df["Cu (%)"],
    weights=composites_df["weight"],
)["IDW declustered"].rename("Declustered Composites")

estimate_summary = (
    estimates_df.groupby("Method")["Cu (%)"]
    .agg(
        count="count",
        min="min",
        mean="mean",
        variance=lambda values: values.var(ddof=0),
        max="max",
    )
    .T
)
summary_df = pd.concat([composite_summary, estimate_summary], axis=1)
summary_df = summary_df.reindex(columns=["Declustered Composites", "Cu_KNN", "Cu_IDW", "Cu_OK"])
summary_df